In [1]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as func

from torchvision.models import resnet18

class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet18(weights=None)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4
    def forward(self, x):
        x0 = self.stem(x)
        x1 = self.encoder1(self.pool(x0))
        x2 = self.encoder2(x1)
        x3 = self.encoder3(x2)
        x4 = self.encoder4(x3)
        return x0, x1, x2, x3, x4


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(skip_channels + out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        skip = func.center_crop(skip, [x.shape[2], x.shape[3]])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ResNetUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        self.encoder = ResNet18Encoder()

        self.decoder4 = DecoderBlock(512, 256, 256)
        self.decoder3 = DecoderBlock(256, 128, 128)
        self.decoder2 = DecoderBlock(128, 64, 64)
        self.decoder1 = DecoderBlock(64, 64, 32)

        self.final_conv = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        x0, x1, x2, x3, x4 = self.encoder(x)

        d4 = self.decoder4(x4, x3)
        d3 = self.decoder3(d4, x2)
        d2 = self.decoder2(d3, x1)
        d1 = self.decoder1(d2, x0)

        out = self.final_conv(d1)
        return out

In [2]:
import torch
from pytorch_wavelets import DWTForward, DWTInverse

wave, levels, mode, device = 'haar', 1, 'reflect', 'cuda'

dwt = DWTForward(J=levels, wave=wave, mode=mode).to(device)
idwt = DWTInverse(wave=wave, mode=mode).to(device)

@torch.no_grad()
def wavelet_adapt(src, tgt, alpha, method='match'):
    src = src.unsqueeze(0)
    tgt = tgt.unsqueeze(0)

    s_LL, s_H = dwt(src)
    t_LL, t_H = dwt(tgt)

    if method == 'replace':
        new_LL = (1 - alpha) * s_LL + alpha * t_LL
    elif method == 'match':
        mu_s, sd_s = s_LL.mean((2,3), keepdim=True), s_LL.std((2,3), keepdim=True).clamp_min(1e-6)
        mu_t, sd_t = t_LL.mean((2,3), keepdim=True), t_LL.std((2,3), keepdim=True).clamp_min(1e-6)
        t_LL_ms = (t_LL - mu_t) / sd_t * sd_s + mu_s
        new_LL = (1 - alpha) * s_LL + alpha * t_LL_ms

    stylized = idwt((new_LL, s_H)).clamp(0,1)
    stylized = stylized.squeeze(0)
    return stylized

In [3]:
from torch.utils.data import Dataset
import tifffile as tif
import random
from torchvision import transforms
import os

random.seed(42)

normalize_05 = transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
to_tensor = transforms.ToTensor()
imageTransform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cloud_thresh = 0.15
max_tries = 10

class LandslideDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
    
    def __len__(self):
        return len(self.img_list)
    
    def __getitem__(self, index):
        img_dir = self.img_list[index]
        mask_dir = self.mask_list[index]

        img = tif.imread(img_dir)
        mask = tif.imread(mask_dir)

        img = imageTransform(img)
        mask = (to_tensor(mask) > 0).float()

        return img, mask
    
class SourceDataset(Dataset):
    def __init__(self, source_ds, target_ds, alpha):
        self.source = source_ds
        self.target = target_ds
        self.alpha = alpha

    def __len__(self):
        return len(self.source)
    
    def sample_target_img(self):
        random.seed()
        tgt_path = random.choice(self.target)
        img = tif.imread(tgt_path)
            
        return to_tensor(img)
    
    def __getitem__(self, idx):
        src = self.source[idx]
        mask = src.replace("img", "mask")

        src_img = to_tensor(tif.imread(src)).float().to(device, non_blocking=True)
        src_mask = (to_tensor(tif.imread(mask)) > 0).float().to(device, non_blocking=True)

        tgt_img = self.sample_target_img().float().to(device, non_blocking=True)

        src_img = wavelet_adapt(src_img, tgt_img, alpha=self.alpha)
        src_img = normalize_05(src_img)

        return src_img, src_mask

path = "../../data/"
limit = 100
regions = [f for f in os.listdir(path) if (os.path.isdir(os.path.join(path, f)) and f != "study areas shp")]

regions_dict = dict()

for region in regions:
    dataset_dir = path + region
    image_dir = os.path.join(dataset_dir, "img")
    img_list = os.listdir(image_dir)

    all_images = sorted(os.path.join(image_dir, f) for f in img_list)

    if len(all_images) > limit:
        all_images = random.sample(all_images, limit)

    regions_dict[region] = all_images

In [4]:
def getMetrics(TP, TN, FP, FN):
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
    iou_1  = TP / (TP + FP + FN + 1e-8)
    iou_0 = TN / (TN + FP + FN + 1e-8)
    miou = (iou_0 + iou_1) / 2 
        
    oa = (TP + TN)/(TP + TN + FP + FN + 1e-8)
    
    return precision, recall, f1, iou_1, miou, oa

def dice_loss(pred, target, smooth=1.):
    pred = torch.sigmoid(pred)
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return 1 - ((2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth))


### COMPUTE R (NEGATIVE PIXELS / POSITIVE PIXELS)
r = 4.5
pos_weight = torch.tensor([r]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [5]:
import torch.nn.functional as F

def train_model(model, optimizer, train_loader, val_loader, epochs, model_path, region_name, threshold=0.6):
    best_iou = 0.0
    patience = 10
    counter = 0

    print(f'region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa')

    for epoch in range(epochs):
        model.train()
        running_loss, n_samples = 0.0, 0

        for images, masks in train_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            bce  = criterion(logits, masks)
            dice = dice_loss(logits, masks)
            loss = bce + dice

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * masks.numel()
            n_samples += masks.numel()
        
        training_loss = running_loss / max(1, n_samples)

        model.eval()
        val_loss, val_num = 0.0, 0
        
        TP, FP, FN, TN = 0,0,0,0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True).float()

                logits = model(images)
                if logits.shape[-2:] != masks.shape[-2:]:
                    logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

                bce  = criterion(logits, masks)
                dice = dice_loss(logits, masks)
                loss = bce + dice
                val_loss += loss.item()

                preds = torch.sigmoid(logits) > threshold

                preds_flat = preds.view(-1)
                mask_flat = masks.view(-1)

                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                
                val_num += 1
             
        precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)

        print(f'{region_name}, {epoch+1}, {training_loss :.3f}, {val_loss / val_num :.3f}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}')

        if iou > best_iou:
            best_iou = iou
            counter = 0
            
            torch.save(model.state_dict(), model_path)
        elif iou != 0.0:
            counter += 1
            if counter >= patience:
                break

def test_model(model, test_loader, threshold=0.6):
    TP, FP, FN, TN = 0,0,0,0

    model.eval()

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            preds = torch.sigmoid(logits) > threshold
                    
            preds_flat = preds.view(-1)
            mask_flat = masks.view(-1)

            TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
            FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
            FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
            TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()

    return getMetrics(TP, TN, FP, FN)

In [6]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

batch_size = 16

output = "source_region,target_region,precision,recall,f1,iou,miou,oa"

alpha = 0.5
alpha_name = "50_replace"

for source_region in regions_dict:
    source_images = regions_dict[source_region]
    source_masks = [f.replace("img", "mask") for f in source_images]

    train_img, val_img, train_mask, val_mask = train_test_split(
        source_images, source_masks, test_size=.2, random_state=42
    )

    for target_region in regions_dict:
        if source_region != target_region:
            epochs = 40

            model = ResNetUNet().to(device)

            target_test_img = regions_dict[target_region]
            target_test_mask = [f.replace("img", "mask") for f in target_test_img]

            train_dataset = SourceDataset(train_img, target_test_img, alpha=alpha)
            val_dataset = LandslideDataset(val_img, val_mask)
            test_dataset = LandslideDataset(target_test_img, target_test_mask)

            trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
            valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)

            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

            model_path = f"../../models/wavelet/{alpha_name}/{source_region}_{target_region}.pth"

            train_model(model, optimizer, trainLoader, valLoader, epochs, model_path, source_region)

            model.load_state_dict(torch.load(model_path, map_location=device))

            precision, recall, f1, iou, miou, oa = test_model(model, testLoader)
            result = f'{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'
            print(result)
            output += f'\n{result}'

with open(f"../../results/wavelet/{alpha_name}.csv", "w") as f:
    f.write(output)


region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa
Hokkaido Iburi-Tobu, 1, 1.744, 1.704, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 2, 1.713, 1.691, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 3, 1.673, 1.662, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 4, 1.619, 1.598, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 5, 1.520, 1.439, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 6, 1.348, 2.958, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 7, 1.228, 1.932, 0.8991, 0.0239, 0.0465, 0.0238, 0.4811, 0.9385
Hokkaido Iburi-Tobu, 8, 1.197, 1.202, 0.8552, 0.4343, 0.5761, 0.4046, 0.6817, 0.9598
Hokkaido Iburi-Tobu, 9, 1.061, 1.115, 0.8163, 0.5381, 0.6486, 0.4800, 0.7210, 0.9634
Hokkaido Iburi-Tobu, 10, 1.090, 0.950, 0.7002, 0.7139, 0.7070, 0.5468, 0.7539, 0.9628
Hokkaido Iburi-Tobu, 11, 1.060, 1.021, 0.5214, 0.8306, 0.6407, 0.4713, 0.70

In [11]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

batch_size = 16

output = "source_region,target_region,precision,recall,f1,iou,miou,oa"

alpha = 0.2
alpha_name = "20_replace"

for source_region in regions_dict:
    source_images = regions_dict[source_region]
    source_masks = [f.replace("img", "mask") for f in source_images]

    train_img, val_img, train_mask, val_mask = train_test_split(
        source_images, source_masks, test_size=.2, random_state=42
    )

    for target_region in regions_dict:
        if source_region != target_region:
            epochs = 40

            model = ResNetUNet().to(device)

            target_test_img = regions_dict[target_region]
            target_test_mask = [f.replace("img", "mask") for f in target_test_img]

            train_dataset = SourceDataset(train_img, target_test_img, alpha=alpha)
            val_dataset = LandslideDataset(val_img, val_mask)
            test_dataset = LandslideDataset(target_test_img, target_test_mask)

            trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
            valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)

            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

            model_path = f"../../models/wavelet/{alpha_name}/{source_region}_{target_region}.pth"

            train_model(model, optimizer, trainLoader, valLoader, epochs, model_path, source_region)

            model.load_state_dict(torch.load(model_path, map_location=device))

            precision, recall, f1, iou, miou, oa = test_model(model, testLoader)
            result = f'{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'
            print(result)
            output += f'\n{result}'

with open(f"../../results/wavelet/{alpha_name}.csv", "w") as f:
    f.write(output)


region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa
Hokkaido Iburi-Tobu, 1, 1.762, 1.712, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 2, 1.741, 1.705, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 3, 1.713, 1.688, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 4, 1.665, 1.645, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 5, 1.570, 1.468, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 6, 1.396, 7.891, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 7, 1.281, 3.219, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 8, 1.198, 1.442, 0.6339, 0.1463, 0.2378, 0.1349, 0.5377, 0.9411
Hokkaido Iburi-Tobu, 9, 1.110, 1.376, 0.7404, 0.3283, 0.4549, 0.2944, 0.6220, 0.9506
Hokkaido Iburi-Tobu, 10, 1.010, 1.175, 0.7918, 0.4944, 0.6087, 0.4375, 0.6982, 0.9601
Hokkaido Iburi-Tobu, 11, 0.967, 0.970, 0.6506, 0.6862, 0.6679, 0.5014, 0.72

In [12]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

batch_size = 16

output = "source_region,target_region,precision,recall,f1,iou,miou,oa"

alpha = 0.3
alpha_name = "30_replace"

for source_region in regions_dict:
    source_images = regions_dict[source_region]
    source_masks = [f.replace("img", "mask") for f in source_images]

    train_img, val_img, train_mask, val_mask = train_test_split(
        source_images, source_masks, test_size=.2, random_state=42
    )

    for target_region in regions_dict:
        if source_region != target_region:
            epochs = 40

            model = ResNetUNet().to(device)

            target_test_img = regions_dict[target_region]
            target_test_mask = [f.replace("img", "mask") for f in target_test_img]

            train_dataset = SourceDataset(train_img, target_test_img, alpha=alpha)
            val_dataset = LandslideDataset(val_img, val_mask)
            test_dataset = LandslideDataset(target_test_img, target_test_mask)

            trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
            valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)

            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

            model_path = f"../../models/wavelet/{alpha_name}/{source_region}_{target_region}.pth"

            train_model(model, optimizer, trainLoader, valLoader, epochs, model_path, source_region)

            model.load_state_dict(torch.load(model_path, map_location=device))

            precision, recall, f1, iou, miou, oa = test_model(model, testLoader)
            result = f'{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'
            print(result)
            output += f'\n{result}'

with open(f"../../results/wavelet/{alpha_name}.csv", "w") as f:
    f.write(output)


region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa
Hokkaido Iburi-Tobu, 1, 1.771, 1.726, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 2, 1.756, 1.721, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 3, 1.733, 1.712, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 4, 1.708, 1.690, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 5, 1.650, 1.600, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 6, 1.507, 2.929, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 7, 1.318, 6.782, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 8, 1.250, 1.501, 0.8058, 0.1738, 0.2859, 0.1668, 0.5558, 0.9455
Hokkaido Iburi-Tobu, 9, 1.257, 1.180, 0.7950, 0.4241, 0.5532, 0.3823, 0.6691, 0.9570
Hokkaido Iburi-Tobu, 10, 1.155, 1.158, 0.8115, 0.5001, 0.6189, 0.4481, 0.7041, 0.9613
Hokkaido Iburi-Tobu, 11, 0.992, 1.108, 0.6805, 0.6771, 0.6788, 0.5137, 0.73